# 基金业绩持续性量化评价

## 研报复现：华泰金工研报《定量评价基金的业绩持续性》

**研究方法**:
1. 横截面分析法 (Cross-Section Analysis)
2. 交叉积比率法 (Cross-Product Ratio, CPR)
3. Hurst指数法 (Hurst Exponent)

---

### 1. 环境配置与数据导入

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 120

import seaborn as sns
plt.style.use('seaborn-v0_8-whitegrid')

from source.data_loader import (
    get_fund_nav,
    calculate_daily_returns,
    calculate_log_returns,
    get_benchmark_data,
    get_multiple_funds_data,
    merge_funds_returns
)
from source.factor import (
    cross_section_single_fund,
    calculate_cpr_single_fund,
    hurst_analysis,
    comprehensive_persistence_analysis
)
from source.plot import (
    plot_persistence_dashboard,
    plot_hurst_distribution,
    plot_cpr_matrix
)
from source.utils import calculate_period_return

print('环境配置完成!')

---

### 2. 设置分析参数

In [ ]:
# 分析参数
FUND_CODE = '000628'  # 大成高鑫股票A
START_DATE = '2018-01-01'
END_DATE = '2023-12-31'
BENCHMARK_CODE = '000300'  # 沪深300

print(f'基金代码: {FUND_CODE}')
print(f'分析期间: {START_DATE} 至 {END_DATE}')
print(f'基准指数: {BENCHMARK_CODE} (沪深300)')

---

### 3. 获取基金数据

In [ ]:
# 获取基金净值数据
print(f'正在获取基金 {FUND_CODE} 净值数据...')
nav_df = get_fund_nav(FUND_CODE, START_DATE, END_DATE)

if nav_df is not None and len(nav_df) > 0:
    print(f'成功获取 {len(nav_df)} 条净值数据')
    print(f'数据范围: {nav_df["date"].min()} 至 {nav_df["date"].max()}')
    display(nav_df.head())
else:
    print('数据获取失败!')

In [ ]:
# 计算收益率
returns_df = calculate_daily_returns(nav_df)
log_returns_df = calculate_log_returns(nav_df)

fund_returns = returns_df.set_index('date')['daily_return']
log_returns = log_returns_df.set_index('date')['log_return']

print(f'日收益率序列长度: {len(fund_returns)}')
print(f'对数收益率序列长度: {len(log_returns)}')
print(f'\n收益率统计:')
print(fund_returns.describe())

In [ ]:
# 获取基准数据
print(f'正在获取基准 {BENCHMARK_CODE} 数据...')
benchmark_df = get_benchmark_data(BENCHMARK_CODE, START_DATE, END_DATE)

if benchmark_df is not None:
    benchmark_returns = benchmark_df.set_index('date')['nav'].pct_change().dropna()
    print(f'成功获取 {len(benchmark_returns)} 条基准收益率数据')
else:
    print('基准数据获取失败，将使用无风险利率计算超额收益')
    benchmark_returns = None

---

### 4. 方法一：横截面分析法

In [ ]:
## 横截面分析法

**原理**: 将样本期划分为两个等长子期间，检验评价期超额收益与持有期超额收益的正相关性

**公式**: α₂ᵢ = α + β × α₁ᵢ

其中:
- α₁ᵢ: 基金i在评价期的超额收益
- α₂ᵢ: 基金i在持有期的超额收益
- β: 持续性系数（若显著为正，说明业绩有持续性）

In [ ]:
# 横截面分析法
cs_result = cross_section_single_fund(
    fund_returns,
    benchmark_returns=benchmark_returns,
    risk_free_rate=0.03
)

print('='*60)
print('横截面分析法结果')
print('='*60)

if cs_result:
    print(f'评价期超额收益 (α₁): {cs_result.get("alpha1", 0):.4f}')
    print(f'持有期超额收益 (α₂): {cs_result.get("alpha2", 0):.4f}')
    print(f'同号判断: {cs_result.get("same_sign", "N/A")}')
    print(f'\n结论: {cs_result.get("persistence", "N/A")}')
else:
    print('分析失败，数据不足')

In [ ]:
# 可视化两个期间的关系
if cs_result:
    fig, ax = plt.subplots(figsize=(8, 8))
    
    # 绘制散点
    ax.scatter([cs_result['alpha1']], [cs_result['alpha2']], 
              s=200, c='blue', alpha=0.7, zorder=5)
    
    # 45度参考线
    max_val = max(abs(cs_result['alpha1']), abs(cs_result['alpha2'])) * 1.2
    ax.plot([-max_val, max_val], [-max_val, max_val], 
           'k--', linewidth=1, alpha=0.5, label='y=x (No persistence)')
    
    # 标记点
    ax.scatter([cs_result['alpha1']], [cs_result['alpha2']], 
              s=200, c='green' if cs_result['same_sign'] else 'red', 
              alpha=0.8, zorder=10)
    
    ax.set_xlabel('Period 1 Excess Return (alpha1)')
    ax.set_ylabel('Period 2 Excess Return (alpha2)')
    ax.set_title(f'Cross-Section Analysis: {FUND_CODE}\nPersistence: {cs_result.get("persistence", "N/A")}')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

---

### 5. 方法二：交叉积比率法 (CPR)

In [ ]:
## 交叉积比率法 (CPR)

**原理**: 将样本期分为多个等长期间，比较赢家和输家的转换概率

**公式**: CPR = (WW × LL) / (WL × LW)

其中:
- WW: 连续两个期间均为赢家
- LL: 连续两个期间均为输家
- WL: 第一期间赢，第二期间输
- LW: 第一期间输，第二期间赢

**判断**:
- CPR ≈ 1: 业绩不具有持续性
- CPR > 1: 业绩有持续性
- CPR < 1: 业绩有反转倾向

In [ ]:
# 交叉积比率法
cpr_result = calculate_cpr_single_fund(
    fund_returns,
    n_periods=4,
    market_median_returns=None
)

print('='*60)
print('交叉积比率法 (CPR) 结果')
print('='*60)

if cpr_result:
    print(f'WW (双赢): {cpr_result.get("WW", 0)}')
    print(f'LL (双输): {cpr_result.get("LL", 0)}')
    print(f'WL (先赢后输): {cpr_result.get("WL", 0)}')
    print(f'LW (先输后赢): {cpr_result.get("LW", 0)}')
    cpr_val = cpr_result.get('CPR', np.nan)
    print(f'\nCPR值: {cpr_val:.4f}' if not np.isnan(cpr_val) else '\nCPR值: 无法计算')
    print(f'结论: {cpr_result.get("persistence_verdict", "N/A")}')

In [ ]:
# 可视化CPR矩阵
if cpr_result:
    fig, ax = plt.subplots(figsize=(8, 8))
    
    matrix = np.array([[cpr_result.get('WW', 0), cpr_result.get('WL', 0)],
                      [cpr_result.get('LW', 0), cpr_result.get('LL', 0)]])
    
    im = ax.imshow(matrix, cmap='YlOrRd', aspect='equal')
    
    labels = [['WW', 'WL'], ['LW', 'LL']]
    for i in range(2):
        for j in range(2):
            color = 'white' if matrix[i, j] > matrix.max() / 2 else 'black'
            ax.text(j, i, f'{labels[i][j]}\n{matrix[i, j]}', 
                   ha='center', va='center', fontsize=18, 
                   color=color, fontweight='bold')
    
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(['Winner (W)', 'Loser (L)'])
    ax.set_yticklabels(['Winner (W)', 'Loser (L)'])
    ax.set_xlabel('Current Period')
    ax.set_ylabel('Next Period')
    
    cpr_val = cpr_result.get('CPR', np.nan)
    if not np.isnan(cpr_val):
        verdict = cpr_result.get('persistence_verdict', 'N/A')
        ax.set_title(f'CPR Matrix: {FUND_CODE}\nCPR = {cpr_val:.2f} ({verdict})')
    else:
        ax.set_title(f'CPR Matrix: {FUND_CODE}\nCPR = N/A')
    
    plt.colorbar(im, ax=ax, shrink=0.8)
    plt.tight_layout()
    plt.show()

---

### 6. 方法三：Hurst指数法

In [ ]:
## Hurst指数法

**原理**: 研究时间序列历史取值对未来取值的影响力（长记忆性）

**公式**: log((R/S)ₙ) = log(c) + H × log(n)

**判断**:
- 0.5 < H < 1: 业绩有正向持续性（越接近1，持续性越强）
- H = 0.5: 收益随机波动，不具备持续性
- 0 < H < 0.5: 业绩有反转倾向（越接近0，反转性越强）

In [ ]:
# Hurst指数分析
hurst_result = hurst_analysis(
    fund_returns,
    log_returns=log_returns,
    n_values=[4, 8, 16, 32],
    n_estimators=8
)

print('='*60)
print('Hurst指数分析结果')
print('='*60)

if hurst_result:
    H = hurst_result.get('H', np.nan)
    print(f'Hurst指数 H: {H:.4f}')
    print(f'常数 c: {hurst_result.get("c", np.nan):.4f}')
    print(f'R方: {hurst_result.get("r_squared", np.nan):.4f}')
    print(f'观测数: {hurst_result.get("n_observations", "N/A")}')
    print(f'\nHurst分类: {hurst_result.get("hurst_category", "N/A")}')
    print(f'结论: {hurst_result.get("persistence_verdict", "N/A")}')

In [ ]:
# 可视化Hurst指数
if hurst_result:
    fig, ax = plt.subplots(figsize=(10, 6))
    
    H = hurst_result.get('H', 0.5)
    
    # 绘制H值仪表
    colors = ['green' if H > 0.5 else 'red']
    bars = ax.barh(['Hurst Index'], [H], color=colors[0], alpha=0.8)
    
    # 标记H=0.5
    ax.axvline(x=0.5, color='gray', linestyle='--', linewidth=2, 
               label='Random (H=0.5)')
    
    # 填充区域
    ax.axvspan(0, 0.4, alpha=0.1, color='red')
    ax.axvspan(0.5, 1, alpha=0.1, color='green')
    
    ax.set_xlim(0, 1)
    ax.set_xlabel('Hurst Index (H)')
    ax.set_title(f'Hurst Index Analysis: {FUND_CODE}')
    
    # 添加数值标签
    ax.text(H, 0, f' H={H:.4f}', va='center', fontsize=14, fontweight='bold')
    
    # 添加分类标签
    verdict = hurst_result.get('persistence_verdict', '')
    color = 'green' if '持续' in verdict else 'red'
    ax.text(0.5, -0.2, verdict, ha='center', fontsize=14, fontweight='bold', 
           color=color, transform=ax.transAxes)
    
    ax.legend(loc='lower right')
    ax.grid(True, alpha=0.3, axis='x')
    
    plt.tight_layout()
    plt.show()

---

### 7. 综合分析仪表盘

In [ ]:
# 构建综合分析结果
analysis_results = {
    '横截面分析法': cs_result,
    '交叉积比率法': cpr_result,
    'Hurst指数法': hurst_result
}

# 计算累计收益率
cumulative_returns = (1 + fund_returns).cumprod()
cumulative_returns.name = FUND_CODE

# 生成综合仪表盘
fig = plot_persistence_dashboard(
    FUND_CODE,
    analysis_results,
    cumulative_returns=cumulative_returns
)

---

### 8. 分析结论

In [ ]:
print('='*60)
print(f'基金 {FUND_CODE} 业绩持续性分析结论')
print('='*60)

# 收集各方法判断
verdicts = []

if cs_result:
    verdicts.append(('横截面分析法', cs_result.get('persistence', 'N/A')))

if cpr_result:
    verdicts.append(('交叉积比率法', cpr_result.get('persistence_verdict', 'N/A')))

if hurst_result:
    verdicts.append(('Hurst指数法', hurst_result.get('persistence_verdict', 'N/A')))

for method, verdict in verdicts:
    print(f'  {method}: {verdict}')

# 综合判断
persistence_count = sum(1 for _, v in verdicts if '持续' in str(v) and '无' not in str(v))
reversal_count = sum(1 for _, v in verdicts if '反转' in str(v))

print(f'\n综合判断:')
if persistence_count >= 2:
    print(f'  业绩整体有持续性 ({persistence_count}/3 方法支持)')
elif reversal_count >= 2:
    print(f'  业绩整体有反转倾向 ({reversal_count}/3 方法支持)')
else:
    print(f'  业绩持续性不明确')

print('\n' + '='*60)
print('分析完成!')
print('='*60)